# USGS dataretrieval Python Package Water-Use Examples

This notebook provides examples of using the Python dataretrieval package to retrieve water-use data from the USGS [National Water Availability Assessment Data Companion (NWDC)](https://water.usgs.gov/nwaa-data/). The NWDC serves national-scale, *modeled* water-use estimates on a HUC12 (12-digit hydrologic unit) grid, queryable for any county, state, or hydrologic unit. It is the modern replacement for the retired legacy water-use service.

The dataretrieval package exposes the service through a single function, `nwdc.get_wateruse`, which returns a tuple of a pandas data frame and an associated metadata object.

### Install the Package

Use the following code to install the package if it doesn't exist already within your Jupyter Python environment.

In [ ]:
!pip install dataretrieval

Load the package so you can use it along with other packages used in this notebook.

In [ ]:
from IPython.display import display

from dataretrieval import nwdc

### Basic Usage

This example uses the `get_wateruse()` function to retrieve modeled water-use estimates. The following arguments are commonly used:

* **model** (string): The water-use category to query, for example `wu-public-supply-wd` (public-supply withdrawals) or `wu-irrigation-wd` (irrigation withdrawals). Available models include:

  | model | description |
  | --- | --- |
  | `wu-public-supply-wd` | public-supply withdrawals |
  | `wu-public-supply-cu` | public-supply consumptive use |
  | `wu-irrigation-wd` | irrigation withdrawals |
  | `wu-irrigation-cu` | irrigation consumptive use |
  | `wu-thermoelectric` | thermoelectric-power water use |

* **variable** (string or list of strings): One or more variable IDs within the chosen model, for example `pswdtot` (total public-supply withdrawals) or `["pswdgw", "pswdsw"]` for the groundwater and surface-water components. The service requires at least one variable.
* **state**, **county**, or **huc** (string, int, or list): Exactly one of these selects the area. `state` accepts a full name (`"Kansas"`), a two-letter postal code (`"KS"`), or a two-digit FIPS code (`"20"`); `county` takes a five-digit FIPS code; and a `huc` code's length sets its hydrologic-unit level.
* **time_resolution** (string): `"monthly"`, `"annualcy"` (annual, calendar year), or `"annualwy"` (annual, water year).
* **start_date**, **end_date** (string): The query window, formatted `"YYYY"` for annual data or `"YYYY-MM"` for monthly data.

See the [NWDC data catalog](https://water.usgs.gov/nwaa-data/) for the full list of models and their variables.

#### Example 1: Retrieve public-supply withdrawals for a State

Request total public-supply withdrawals for every HUC12 in Kansas, as annual calendar-year values. Because a State spans many HUC12s, the service returns the result across several pages; `get_wateruse()` follows the pagination links and stitches the pages into one frame.

In [ ]:
kansas, md = nwdc.get_wateruse(
    model="wu-public-supply-wd",
    variable="pswdtot",
    state="KS",
    start_date="2015",
    end_date="2020",
    time_resolution="annualcy",
)
print(
    "Retrieved "
    + str(len(kansas))
    + " records across "
    + str(kansas["huc12_id"].nunique())
    + " HUC12 watersheds."
)

### Interpreting the Result

The `get_wateruse()` function returns a tuple containing a pandas data frame and an associated metadata object. The data frame is in long (tidy) form: one row per HUC12 and time step. The `huc12_id` column is kept as a string so leading zeros are preserved, and each value column carries its unit — here `_mgd`, million gallons per day.

Once you've got the data frame, there are several useful things you can do to explore the data.

In [ ]:
# Display the data frame as a table
display(kansas)

Show the data types of the columns in the resulting data frame.

In [ ]:
print(kansas.dtypes)

Aggregate the HUC12 grid to a statewide total for each year.

In [ ]:
statewide = kansas.groupby("year")["pswdtot_mgd"].sum()
display(statewide)

The other part of the result returned from the `get_wateruse()` function is a metadata object that contains information about the query that was executed to return the data. For example, you can access the URL that was assembled to retrieve the requested data from the NWDC.

In [ ]:
print("The query URL used to retrieve the data from the NWDC was: " + md.url)

### Additional Examples

#### Example 2: Split public-supply withdrawals by source

Request the total, groundwater, and surface-water components of public-supply withdrawals for Rhode Island as monthly values for 2020. Passing a list of variables returns one value column per variable.

In [ ]:
rhode_island, md = nwdc.get_wateruse(
    model="wu-public-supply-wd",
    variable=["pswdtot", "pswdgw", "pswdsw"],
    state="RI",
    start_date="2020-01",
    end_date="2020-12",
    time_resolution="monthly",
)
display(rhode_island)

#### Example 3: Retrieve a different water-use category

Swap the `model` and `variable` to retrieve irrigation withdrawals instead of public supply. Here we request total irrigation withdrawals (`irrwdtot`) for Nebraska as annual calendar-year values.

In [ ]:
nebraska, md = nwdc.get_wateruse(
    model="wu-irrigation-wd",
    variable="irrwdtot",
    state="NE",
    start_date="2015",
    end_date="2020",
    time_resolution="annualcy",
)
display(nebraska)

### Notes

- **Spatial resolution.** Estimates are always returned on the HUC12 grid, regardless of the area you query. Joining `huc12_id` to a HUC12 boundary layer (for example with `geopandas`) lets you map the results.
- **Pick one selector.** Use exactly one of `state`, `county` (five-digit FIPS), or `huc`. Each selector also accepts a list (e.g. `state=["NE", "KS"]`), fanned out into one request per area and concatenated.
- **Large areas paginate.** A query spanning more than `limit` HUC12s (600 by default) is split across pages; `get_wateruse()` follows the links and returns a single concatenated frame.
- **Time columns.** Monthly data come back with a `year_month` column; annual data (`annualcy` or `annualwy`) come back with a `year` column.